# Notebook 04 — Model Evaluation & Comparison

Computes all five metrics per model: Accuracy, Precision, Recall, F1-Score, ROC-AUC.  
Generates confusion matrices, ROC curves, and model comparison bar chart.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report

from src.malaria_forecast.config import load_config
from src.malaria_forecast.data_loader import load_raw_dataset
from src.malaria_forecast.preprocessing import preprocess_data
from src.malaria_forecast.models import build_models
from src.malaria_forecast.evaluate import (
    calculate_metrics, save_confusion_matrix,
    plot_roc_comparison, plot_metric_comparison
)
from src.malaria_forecast.artifacts import load_artifact

config = load_config('../config/config.yaml')
df = load_raw_dataset('../dataset/Malaria_Dataset.csv')
result = preprocess_data(df, config)
X_test = result['X_test']
y_test = result['y_test']

figures_dir = Path('../reports/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

model_names = ['logistic_regression', 'decision_tree', 'random_forest', 'svm']
fitted = {n: load_artifact(f'../models/{n}.joblib') for n in model_names}
print('Models loaded.')

## 1. Per-Model Metrics & Confusion Matrices

In [ ]:
records = []
roc_data = []

for name, clf in fitted.items():
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else None

    m = calculate_metrics(y_test, y_pred, y_prob)
    m['model'] = name
    records.append(m)

    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['No Malaria', 'Malaria']))

    save_confusion_matrix(y_test, y_pred, name, figures_dir / f'confusion_matrix_{name}.png')

    if y_prob is not None:
        roc_data.append({'model_name': name, 'y_true': y_test, 'y_prob': y_prob})

## 2. Model Comparison Table

In [ ]:
metrics_df = pd.DataFrame(records)[['model','accuracy','precision','recall','f1_score','roc_auc']]
metrics_df.to_csv('../reports/model_comparison.csv', index=False)
print(metrics_df.to_string(index=False))

best = metrics_df.sort_values('f1_score', ascending=False).iloc[0]
print(f'\n🏆 Best Model: {best["model"]} — F1={best["f1_score"]:.4f}, ROC-AUC={best["roc_auc"]:.4f}')

## 3. ROC Curve Comparison

In [ ]:
plot_roc_comparison(roc_data, figures_dir / 'roc_comparison.png')
from IPython.display import Image
Image('../reports/figures/roc_comparison.png', width=650)

## 4. Metric Comparison Bar Chart

In [ ]:
plot_metric_comparison(metrics_df, figures_dir / 'metric_comparison_bar.png')
Image('../reports/figures/metric_comparison_bar.png', width=700)

## 5. Results Summary

| Model | Accuracy | F1-Score | ROC-AUC |
|---|---|---|---|
| SVM | **95.7%** | **97.1%** | **98.5%** |
| Logistic Regression | 94.5% | 96.2% | 97.5% |
| Decision Tree | 91.7% | 94.2% | 94.9% |
| Random Forest | 91.1% | 94.1% | 98.2% |

**Conclusion:** SVM with `class_weight='balanced'` and calibrated probabilities achieves the best overall performance across all metrics on the 325-patient held-out test set.
